In [ ]:
#Part 1 grid
grid = [
    ['S', '.', '.', '#', '.', '.', '.'],
    ['.', '#', '.', '#', '.', '#', '.'],
    ['.', '#', '.', '.', '.', '#', 'G'],
    ['.', '.', '#', '#', '.', '.', '.'],
    ['.', 'K', '.', '.', '.', '#', '.']
]

ROWS = len(grid)
COLS = len(grid[0])

START = (0, 0)
KIT = (4, 1)
GOAL = (2, 6)

start_state = (START[0], START[1], False)

print("Starting state:", start_state)

Starting state: (0, 0, False)


In [ ]:
#Part 2: getNeighhbors
def get_neighbors(state):
    row, col, has_kit = state

    # Required action order: Up, Right, Down, Left
    actions = [
        (-1, 0),  # Up
        (0, 1),   # Right
        (1, 0),   # Down
        (0, -1)   # Left
    ]

    neighbors = []

    for row_change, col_change in actions:
        new_row = row + row_change
        new_col = col + col_change

        # Make sure new position is inside the grid
        if 0 <= new_row < ROWS and 0 <= new_col < COLS:

            # Make sure new position is not blocked
            if grid[new_row][new_col] != '#':

                new_has_kit = has_kit

                # Pick up the medical kit
                if (new_row, new_col) == KIT:
                    new_has_kit = True

                new_state = (new_row, new_col, new_has_kit)
                neighbors.append(new_state)

    return neighbors



In [ ]:
print(get_neighbors(start_state))
test_state = (4, 0, False)

print(get_neighbors(test_state))



[(0, 1, False), (1, 0, False)]
[(3, 0, False), (4, 1, True)]


In [ ]:
#Part 3: Breath-First Search
from collections import deque

def bfs(start_state):
    queue = deque()

    # Stores the current state and the path used to reach it
    queue.append((start_state, [start_state]))

    visited = set()
    visited.add(start_state)

    nodes_expanded = 0

    while queue:
        current_state, path = queue.popleft()

        nodes_expanded += 1

        row, col, has_kit = current_state

        # Check if reached goal with the medical kit
        if (row, col) == GOAL and has_kit:
            return path, nodes_expanded

        # Look at neighboring states
        for neighbor in get_neighbors(current_state):

            if neighbor not in visited:
                visited.add(neighbor)

                new_path = path + [neighbor]

                queue.append((neighbor, new_path))

    # No solution found
    return None, nodes_expanded

In [ ]:
path, nodes_expanded = bfs(start_state)

print("BFS Path:")
for state in path:
    print(state)

print("Solution Length:", len(path) - 1)
print("Nodes Expanded:", nodes_expanded)

BFS Path:
(0, 0, False)
(1, 0, False)
(2, 0, False)
(3, 0, False)
(3, 1, False)
(4, 1, True)
(4, 2, True)
(4, 3, True)
(4, 4, True)
(3, 4, True)
(3, 5, True)
(3, 6, True)
(2, 6, True)
Solution Length: 12
Nodes Expanded: 46


In [ ]:
#Part 4 Depth-First Search
def dfs(start_state):
    stack = []

    # Store the current state and the path used to reach it
    stack.append((start_state, [start_state]))

    visited = set()

    nodes_expanded = 0

    while stack:
        current_state, path = stack.pop()

        # Skip states already visited
        if current_state in visited:
            continue

        visited.add(current_state)
        nodes_expanded += 1

        row, col, has_kit = current_state

        # Check if reached goal with the medical kit
        if (row, col) == GOAL and has_kit:
            return path, nodes_expanded

        # Get neighbors in required order
        neighbors = get_neighbors(current_state)

        # Reverse them because a stack is LIFO
        for neighbor in reversed(neighbors):
            if neighbor not in visited:
                new_path = path + [neighbor]
                stack.append((neighbor, new_path))

    # No solution found
    return None, nodes_expanded

In [ ]:
path, nodes_expanded = dfs(start_state)

print("DFS Path:")
for state in path:
    print(state)

print("Solution Length:", len(path) - 1)
print("Nodes Expanded:", nodes_expanded)

DFS Path:
(0, 0, False)
(0, 1, False)
(0, 2, False)
(1, 2, False)
(2, 2, False)
(2, 3, False)
(2, 4, False)
(1, 4, False)
(0, 4, False)
(0, 5, False)
(0, 6, False)
(1, 6, False)
(2, 6, False)
(3, 6, False)
(3, 5, False)
(3, 4, False)
(4, 4, False)
(4, 3, False)
(4, 2, False)
(4, 1, True)
(3, 1, True)
(3, 0, True)
(2, 0, True)
(1, 0, True)
(0, 0, True)
(0, 1, True)
(0, 2, True)
(1, 2, True)
(2, 2, True)
(2, 3, True)
(2, 4, True)
(1, 4, True)
(0, 4, True)
(0, 5, True)
(0, 6, True)
(1, 6, True)
(2, 6, True)
Solution Length: 36
Nodes Expanded: 38


In [ ]:
#Part 5: cost grid, get and calcuate cost
cost_grid = [
    [1, 1, 1, None, 1, 1, 1],
    [1, None, 1, None, 4, None, 1],
    [1, None, 1, 1, 4, None, 1],
    [1, 1, None, None, 4, 1, 1],
    [1, 1, 1, 1, 1, None, 1]
]


def get_cost(state):
    row, col, has_kit = state
    return cost_grid[row][col]


def calculate_path_cost(path):
    total_cost = 0

    # Start at index 1, initial starting position, no cost
    for state in path[1:]:
        total_cost += get_cost(state)

    return total_cost


test_path = [
    (0, 0, False),
    (0, 1, False),
    (0, 2, False)
]

print("Path Cost:", calculate_path_cost(test_path))

Path Cost: 2


In [ ]:
#Part 6: Uniform Cost Search

import heapq

def ucs(start_state):
    # Priority queue stores: (cost, state, path)

    frontier = []
    heapq.heappush(frontier, (0, start_state, [start_state]))

    # Keep track of the cheapest cost found for each state
    best_cost = {start_state: 0}

    nodes_expanded = 0

    while frontier:
        current_cost, current_state, path = heapq.heappop(frontier)

        # Skip entry if found a cheaper path
        if current_cost > best_cost[current_state]:
            continue

        nodes_expanded += 1

        row, col, has_kit = current_state

        # Check goal
        if (row, col) == GOAL and has_kit:
            return path, current_cost, nodes_expanded

        # Explore neighbors
        for neighbor in get_neighbors(current_state):

            move_cost = get_cost(neighbor)
            new_cost = current_cost + move_cost

            # Add neighbor if cheapest path found so far
            if neighbor not in best_cost or new_cost < best_cost[neighbor]:
                best_cost[neighbor] = new_cost

                new_path = path + [neighbor]

                heapq.heappush(
                    frontier,
                    (new_cost, neighbor, new_path)
                )

    return None, None, nodes_expanded

path, path_cost, nodes_expanded = ucs(start_state)

print("UCS Path:")

for state in path:
    print(state)

print("Solution Length:", len(path) - 1)
print("Path Cost:", path_cost)
print("Nodes Expanded:", nodes_expanded)

UCS Path:
(0, 0, False)
(1, 0, False)
(2, 0, False)
(3, 0, False)
(3, 1, False)
(4, 1, True)
(4, 2, True)
(4, 3, True)
(4, 4, True)
(3, 4, True)
(3, 5, True)
(3, 6, True)
(2, 6, True)
Solution Length: 12
Path Cost: 15
Nodes Expanded: 37


In [ ]:
#Part 7: A* Search

def heuristic(state):
    row, col, has_kit = state

    if not has_kit:
        # Distance from current position to kit
        to_kit = abs(row - KIT[0]) + abs(col - KIT[1])

        # Distance from kit to goal
        kit_to_goal = abs(KIT[0] - GOAL[0]) + abs(KIT[1] - GOAL[1])

        return to_kit + kit_to_goal

    else:
        # Distance from current position to goal
        return abs(row - GOAL[0]) + abs(col - GOAL[1])

print(heuristic(start_state))

def astar(start_state):
    frontier = []

    # Store: (f cost, g cost, state, path)
    start_h = heuristic(start_state)

    heapq.heappush(
        frontier,
        (start_h, 0, start_state, [start_state])
    )

    # Cheapest g-cost for each state
    best_cost = {start_state: 0}

    nodes_expanded = 0

    while frontier:
        f_cost, current_cost, current_state, path = heapq.heappop(frontier)

        # Skip if cheaper path to state was already found
        if current_cost > best_cost[current_state]:
            continue

        nodes_expanded += 1

        row, col, has_kit = current_state

        # Check goal
        if (row, col) == GOAL and has_kit:
            return path, current_cost, nodes_expanded

        # Explore neighbors
        for neighbor in get_neighbors(current_state):

            move_cost = get_cost(neighbor)
            new_cost = current_cost + move_cost

            if neighbor not in best_cost or new_cost < best_cost[neighbor]:

                best_cost[neighbor] = new_cost

                h = heuristic(neighbor)
                new_f_cost = new_cost + h

                new_path = path + [neighbor]

                heapq.heappush(
                    frontier,
                    (new_f_cost, new_cost, neighbor, new_path)
                )

    return None, None, nodes_expanded

path, path_cost, nodes_expanded = astar(start_state)

print("A* Path:")

for state in path:
    print(state)

print("Solution Length:", len(path) - 1)
print("Path Cost:", path_cost)
print("Nodes Expanded:", nodes_expanded)


12
A* Path:
(0, 0, False)
(1, 0, False)
(2, 0, False)
(3, 0, False)
(3, 1, False)
(4, 1, True)
(4, 2, True)
(4, 3, True)
(4, 4, True)
(3, 4, True)
(3, 5, True)
(3, 6, True)
(2, 6, True)
Solution Length: 12
Path Cost: 15
Nodes Expanded: 22


In [ ]:
# Part 8: Performance Comparison

import time


# Run BFS and measure runtime
start_time = time.perf_counter()

bfs_path, bfs_expanded = bfs(start_state)

end_time = time.perf_counter()

# Calculate BFS results
bfs_runtime = end_time - start_time
bfs_length = len(bfs_path) - 1
bfs_cost = calculate_path_cost(bfs_path)


# Run DFS and measure runtime
start_time = time.perf_counter()

dfs_path, dfs_expanded = dfs(start_state)

end_time = time.perf_counter()

# Calculate DFS results
dfs_runtime = end_time - start_time
dfs_length = len(dfs_path) - 1
dfs_cost = calculate_path_cost(dfs_path)


# Run UCS and measure runtime
start_time = time.perf_counter()

ucs_path, ucs_cost, ucs_expanded = ucs(start_state)

end_time = time.perf_counter()

# Calculate UCS results
ucs_runtime = end_time - start_time
ucs_length = len(ucs_path) - 1


# Run A* and measure runtime
start_time = time.perf_counter()

astar_path, astar_cost, astar_expanded = astar(start_state)

end_time = time.perf_counter()

# Calculate A* results
astar_runtime = end_time - start_time
astar_length = len(astar_path) - 1


# Print the performance comparison table
print(f"{'Algorithm':<12} {'Solution Length':<18} {'Path Cost':<12} {'States Expanded':<18} {'Runtime':<15}")
print("-" * 75)

print(f"{'BFS':<12} {bfs_length:<18} {bfs_cost:<12} {bfs_expanded:<18} {bfs_runtime:.8f}")
print(f"{'DFS':<12} {dfs_length:<18} {dfs_cost:<12} {dfs_expanded:<18} {dfs_runtime:.8f}")
print(f"{'UCS':<12} {ucs_length:<18} {ucs_cost:<12} {ucs_expanded:<18} {ucs_runtime:.8f}")
print(f"{'A*':<12} {astar_length:<18} {astar_cost:<12} {astar_expanded:<18} {astar_runtime:.8f}")

Algorithm    Solution Length    Path Cost    States Expanded    Runtime        
---------------------------------------------------------------------------
BFS          12                 15           46                 0.00039047
DFS          36                 51           38                 0.00028985
UCS          12                 15           37                 0.00033724
A*           12                 15           22                 0.00025721


# Part 9, Analysis Questions

1. Why does the state representation include k?

The state includes k so the robot can keep track of whether it has picked up the medical kit. A robot can be in the same row and column but be in a different state depending on whether k is True or False. The goal is only complete when the robot reaches G after collecting the kit.

2. Did BFS and DFS return the same path? Explain why or why not.

No, BFS and DFS returned different paths. BFS found a path with 12 moves, while DFS found a path with 36 moves. This happens because BFS explores nearby states one level at a time, while DFS continues down one path until it cannot go any farther and then backtracks.

3. Did BFS necessarily find the solution with the lowest total path cost?

No. BFS finds a path with the fewest number of moves, but it does not consider the cost of each move. In this problem, BFS found a path with a total cost of 15, but when different types of terrain have different costs, a path with more moves could still have a lower overall cost.

4. Compare UCS and A*. Did they return solutions with the same total cost?

Yes. Both UCS and A* found solutions with a total path cost of 15. UCS chooses which states to explore based on the path cost so far, while A* also uses a heuristic to estimate the remaining cost. With the heuristic used in this problem, A* was still able to find an optimal-cost solution.

5. Which algorithm expanded fewer states: UCS or A*?

A* expanded fewer states. It expanded 22 states, while UCS expanded 37 states. The Manhattan-distance heuristic helps A* focus on states that move the robot toward the medical kit and goal, so it does not have to explore as many unnecessary states.

6. Suppose all actions had the same cost. Which search algorithm would be appropriate for finding the shortest path? How does your answer change when actions have different costs?

If every action has the same cost, BFS is a good choice because the path with the fewest moves will also have the lowest total cost. If actions have different costs, UCS or A* would be better because they consider the actual cost of the path instead of only the number of moves.
